In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score


import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [4]:
encoder = LabelEncoder()
normalize = MinMaxScaler()

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=numpy.array(train[i[0]])
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

In [6]:
train = train.fillna(0)
test = test.fillna(0)

train_x = train.drop(columns=['efs'])
train_y = train['efs']

for i in test:
    test[i]=test[i].to_numpy()
train.head()

,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,...,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs
0,7,0,7,0,0.0,0.0,0,0,6.0,0,...,0,4,2,1,8.0,0,2.0,0,10.0,0.0
1,2,0,1,0,2.0,8.0,6,0,6.0,1,...,0,3,1,1,8.0,0,2.0,2,10.0,1.0
2,7,0,7,0,2.0,8.0,0,0,6.0,0,...,0,3,1,1,8.0,0,2.0,0,10.0,0.0
3,0,0,1,0,2.0,8.0,0,0,6.0,0,...,2,3,2,1,8.0,0,2.0,0,10.0,0.0
4,0,0,7,0,2.0,8.0,0,0,6.0,1,...,0,3,1,0,8.0,0,2.0,0,10.0,0.0


In [7]:
lgbm_hyperparams={ 'colsample_bytree': 0.3394676841949491,
                   'drop_rate': 0.005229247069083676,
                   'learning_rate': 0.08939376710901481,
                   'max_bin': 3192,
                   'max_depth': 5655,
                   'max_drop': 3654,
                   'min_child_samples': 9011,
                   'min_data_in_leaf': 1320,
                   'n_estimators': 5619,
                   'num_leaves': 8002,
                   'objective': 'regression_l1',
                   'reg_alpha': 0.3092258349709437,
                   'reg_lambda': 0.5130123398875309,
                   'skip_drop': 0.5672900003846416,
                   'verbosity': -1}

XGB_hyperparams = {'colsample_bytree': 0.6679438268550072,
                   'gamma': 2,
                   'learning_rate': 0.19750405984350627,
                   'max_depth': 20,
                   'min_child_weight': 9,
                   'n_estimators': 2338,
                   'objective': 'binary:logistic',
                   'reg_alpha': 44,
                   'reg_lambda': 0.5688906203980144,
                   'subsample': 0.5716652066840949}

In [8]:
kf = KFold(n_splits=5)

LGBM_pre_test = numpy.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print(f"FOLD: {i}")

    x_fold=train_x[train_index[0]: train_index[len(train_index)-1]]
    y_fold=train_y[train_index[0]: train_index[len(train_index)-1]]

    x_test_fold=train_x[test_index[0]: test_index[len(test_index)-1]]
    y_test_fold=train_y[test_index[0]: test_index[len(test_index)-1]]

    lightgbm = AdaBoostRegressor(LGBMRegressor(**lgbm_hyperparams),
                                 random_state=42, 
                                 n_estimators=4,
                                 learning_rate=0.1)
    lightgbm.fit(x_fold, y_fold)

    LGBM_pre_test+=lightgbm.predict(test)

    print(lightgbm.score(x_test_fold.to_numpy(), y_test_fold.to_numpy()))

FOLD: 0
0.08665353495245831
FOLD: 1
0.14058996993120798
FOLD: 2
0.14137566059408202
FOLD: 3
0.12618919791532213
FOLD: 4
0.039697278056633256


In [9]:
kf = KFold(n_splits=5)

xgb_pre_test = numpy.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print(f"FOLD: {i}")

    x_fold=train_x[train_index[0]: train_index[len(train_index)-1]]
    y_fold=train_y[train_index[0]: train_index[len(train_index)-1]]

    x_test_fold=train_x[test_index[0]: test_index[len(test_index)-1]]
    y_test_fold=train_y[test_index[0]: test_index[len(test_index)-1]]

    xgboost = AdaBoostRegressor(XGBRegressor(n_estimators=int(XGB_hyperparams['n_estimators']),  
                                         learning_rate=XGB_hyperparams['learning_rate'],
                                         colsample_bytree=XGB_hyperparams['colsample_bytree'], 
                                         subsample=XGB_hyperparams['subsample'], 
                                         objective='binary:logistic',
                                         min_child_weight=XGB_hyperparams['min_child_weight'],
                                         gamma=XGB_hyperparams['gamma'],
                                         max_depth=int(XGB_hyperparams['max_depth']),
                                         reg_alpha=XGB_hyperparams['reg_alpha'],
                                         reg_lambda=XGB_hyperparams['reg_lambda']),
                                random_state=42, 
                                n_estimators=4,
                                learning_rate=0.1)
    xgboost.fit(x_fold, y_fold)

    xgb_pre_test+=xgboost.predict(test)

    print(xgboost.score(x_test_fold.to_numpy(), y_test_fold.to_numpy()))

FOLD: 0
0.2000588978605663
FOLD: 1
0.21649318416693863
FOLD: 2
0.22518129956902766
FOLD: 3
0.21907692785970034
FOLD: 4
0.19322226895148187


In [10]:
kf = KFold(n_splits=10)

cat_pre_test = numpy.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print(f"FOLD: {i}")

    x_fold=train_x[train_index[0]: train_index[len(train_index)-1]]
    y_fold=train_y[train_index[0]: train_index[len(train_index)-1]]

    x_test_fold=train_x[test_index[0]: test_index[len(test_index)-1]]
    y_test_fold=train_y[test_index[0]: test_index[len(test_index)-1]]

    catboost = CatBoostRegressor(learning_rate=0.009,
                                 depth=5,
                                 l2_leaf_reg=3.5,
                                 min_child_samples=32,
                                 grow_policy='Depthwise',
                                 iterations=8000,
                                 eval_metric='RMSE',
                                 od_type='Iter',
                                 od_wait=50,
                                 random_state=42,
                                 logging_level='Silent')
    catboost.fit(x_fold, y_fold)

    cat_pre_test+=catboost.predict(test)

    print(catboost.score(x_test_fold.to_numpy(), y_test_fold.to_numpy()))

FOLD: 0
0.18534865215664842
FOLD: 1
0.47531458578089125
FOLD: 2
0.46636525221439973
FOLD: 3
0.4723288588413296
FOLD: 4
0.4804431813471107
FOLD: 5
0.4728530832496972
FOLD: 6
0.46212634072002734
FOLD: 7
0.46890377326501964
FOLD: 8
0.4679852405573264
FOLD: 9
0.1850354447856014


In [11]:
RandomForest = {'criterion': 'poisson',
                'max_depth': 7,
                'max_features': 'sqrt',
                'max_leaf_nodes': 5,
                'min_samples_leaf': 8,
                'min_samples_split': 19,
                'min_weight_fraction_leaf': 0.01391485916360663,
                'random_state': 90}

kf = KFold(n_splits=10)

RandomF_pre_test = numpy.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print(f"FOLD: {i}")

    x_fold=train_x[train_index[0]: train_index[len(train_index)-1]]
    y_fold=train_y[train_index[0]: train_index[len(train_index)-1]]

    x_test_fold=train_x[test_index[0]: test_index[len(test_index)-1]]
    y_test_fold=train_y[test_index[0]: test_index[len(test_index)-1]]

    RandomF = RandomForestRegressor()
    RandomF.fit(x_fold, y_fold)

    RandomF_pre_test += RandomF.predict(test)

    print(RandomF.score(x_test_fold.to_numpy(), y_test_fold.to_numpy()))

FOLD: 0
0.1473600322654205
FOLD: 1
0.8819271080366028
FOLD: 2
0.8818813682041713
FOLD: 3
0.878371642826351
FOLD: 4
0.8825526415781222
FOLD: 5
0.8798544908524253
FOLD: 6
0.8806094775197905
FOLD: 7
0.8788970190794402
FOLD: 8
0.8803847193608703
FOLD: 9
0.13163089167363828


In [12]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']
test_predictions = ((RandomF_pre_test/10)+(LGBM_pre_test/5)+(xgb_pre_test/5)+(cat_pre_test/5))/4
test_predictions

array([0.22680191, 0.9735339 , 0.1985584 ])

In [13]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.226802
1,28801,0.973534
2,28802,0.198558


In [14]:
submission.to_csv('submission.csv', index = False)